# 第 7 周 - 笔记本 1：数据准备

## 练习目标（理念）

为 **Llama 3.2** 微调准备训练数据（对应课程第 7 周：开源模型微调）。你将完成：

1. 从 **HuggingFace Hub** 加载商品数据集
2. 分析 **token（代币/词元）** 长度分布，决定截断阈值
3. 把每条样本做成 **prompt–completion** 对（提示 → 价格补全）
4. 把整理好的数据上传回 HuggingFace Hub，供后续训练笔记本使用

## 怎么跑

1. 确保上级目录有 `src/`（含 `items.py`、`config.py`）与 `.env`（含 `HF_TOKEN`）
2. 从上到下依次运行单元格（Shift+Enter）
3. 预计 **10–15 分钟**（上传步骤视网速而定）


In [ ]:
# ========== 导入与鉴权：把数据准备要用的工具箱搬进来 ==========

# 导入标准库 sys：用来改 Python 模块搜索路径（sys.path）
import sys
# 把上一级目录加入 path，才能 import 项目里的 src 包（本笔记本在 notebooks/ 下）
sys.path.append('..')

# 导入标准库 os：读环境变量（Environment Variables），例如 HF_TOKEN
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# 从 huggingface_hub 导入 login：用 token 登录 Hugging Face，才能拉/推私有或限流资源
from huggingface_hub import login
# 从 transformers 导入 AutoTokenizer：按模型名自动加载对应分词器（Tokenizer）
from transformers import AutoTokenizer
# 从 tqdm.notebook 导入进度条：在 Jupyter 里显示循环进度
from tqdm.notebook import tqdm
# 导入 matplotlib.pyplot：画 token 分布直方图
import matplotlib.pyplot as plt

# 从项目 src 导入 Item：封装单条商品样本及 prompt 构造逻辑
from src.items import Item
# 从项目 src 导入 config：集中存放数据集名、基座模型名、MAX_TOKENS 等配置
from src.config import config

# 加载 .env：把 HF_TOKEN 等读入 os.environ（参数保持原样，不擅自加 override）
load_dotenv()
# 从环境变量取出 Hugging Face token（键名必须是 HF_TOKEN）
hf_token = os.environ['HF_TOKEN']
# 登录 HF；add_to_git_credential=True 便于后续 git/LFS 相关操作复用凭证
login(hf_token, add_to_git_credential=True)

# 打印确认：环境与登录就绪（文案保持原样）
print("✅ Environment loaded")


## 显示配置

先看一眼 `config` 里的关键常量：数据集名、基座模型、`MAX_TOKENS` / `MAX_SEQ_LENGTH` 等。后面单元格都依赖这些值。


In [ ]:
# 调用配置对象的 display：把当前超参/路径打印出来，便于核对是否指对仓库与模型
config.display()


## 步骤 1：加载数据集

从 HuggingFace Hub 拉取已处理好的商品数据，并拆成 **train / val / test** 三份。


In [ ]:
# 打印即将加载的数据集名称（来自 config.DATASET_NAME）
print(f"Loading dataset: {config.DATASET_NAME}")
# Item.from_hub：从 Hub 下载并解析为 Item 列表，返回 train、val、test 三个列表
train, val, test = Item.from_hub(config.DATASET_NAME)
# 合并三份：后面做全量 token 统计时用 items（训练本身仍用拆好的子集）
items = train + val + test

# 打印加载结果摘要（文案与格式保持原样）
print(f"\n✅ Loaded:")
print(f"   Training: {len(train):,} items")
print(f"   Validation: {len(val):,} items")
print(f"   Test: {len(test):,} items")
print(f"   Total: {len(items):,} items")


## 步骤 2：分析 token 分布

用基座模型的 **Tokenizer** 统计每条商品 summary 的 token 数，决定截断阈值 `MAX_TOKENS` 是否合理。


In [ ]:
# 打印将要加载的分词器对应的基座模型名
print(f"Loading tokenizer: {config.BASE_MODEL}")
# AutoTokenizer.from_pretrained：按模型 id 从 Hub 拉取分词器（与后续微调模型一致很重要）
tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
# 确认分词器已就绪
print("✅ Tokenizer loaded")


In [ ]:
# 提示：开始统计每条 summary 的 token 数
print("Counting tokens in product summaries...")
# 列表推导：对每个 Item 调用 count_tokens(tokenizer)；tqdm 显示进度
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]

# 平均 token 数：总和 / 样本数
avg_tokens = sum(token_counts) / len(token_counts)
# 最长样本的 token 数
max_tokens = max(token_counts)

# 打印统计摘要（文案保持原样）
print(f"\n📊 Token Statistics:")
print(f"   Average: {avg_tokens:.1f} tokens")
print(f"   Maximum: {max_tokens} tokens")


In [ ]:
# 新建画布：宽 15、高 6，便于看清长尾分布
plt.figure(figsize=(15, 6))
# 标题里带上均值与最大值，方便一眼对照
plt.title(f"Token Distribution: Avg {avg_tokens:.1f}, Max {max_tokens}")
# X 轴：summary 的 token 个数
plt.xlabel('Number of tokens in summary')
# Y 轴：落入每个区间的样本个数
plt.ylabel('Count')
# 直方图：bins 从 0 到 200、步长 10；颜色与柱宽保持原样
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
# 竖线标出 config.MAX_TOKENS：超过该线的样本后续会被截断
plt.axvline(config.MAX_TOKENS, color='red', linestyle='--', label=f'Cutoff: {config.MAX_TOKENS}')
# 显示图例（含 Cutoff 标签）
plt.legend()
# 在笔记本中渲染图像
plt.show()


In [ ]:
# 统计超过 MAX_TOKENS 的样本数：这些在做 prompt 时会被截断
truncated = len([count for count in token_counts if count > config.MAX_TOKENS])
# 截断比例（百分比）
truncated_pct = truncated / len(items) * 100

# 打印截断影响（文案与计算保持原样）
print(f"\n📏 With cutoff of {config.MAX_TOKENS} tokens:")
print(f"   {truncated:,} items will be truncated ({truncated_pct:.1f}%)")
print(f"   {len(items) - truncated:,} items fit within limit ({100-truncated_pct:.1f}%)")


## 步骤 3：创建 prompt–completion 对

把「商品描述 → 价格」做成监督微调可用的 **prompt / completion**。训练/验证常把价格取整，测试集保留精确价格以便评估。


In [ ]:
# 对照示例：先看「尚未生成 prompt」时的原始字段
print("📝 Example item BEFORE prompt creation:")
# 打印训练集第一条的标题
print(f"\nTitle: {train[0].title}")
# 打印真实价格（两位小数）
print(f"Price: ${train[0].price:.2f}")
# 打印 summary 前 200 字符，避免刷屏
print(f"\nSummary:\n{train[0].summary[:200]}...")


In [ ]:
# 提示：开始为全部样本构造 prompt–completion
print("Creating prompt-completion pairs...")

# 训练 + 验证：make_prompts(..., do_round=True) —— 价格取整，降低标签噪声、利于学习
for item in tqdm(train + val, desc="Train/Val"):
    item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=True)

# 测试集：do_round=False —— 保留精确价格，评估时才能算真实误差
for item in tqdm(test, desc="Test"):
    item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)

# 确认构造完成
print("✅ Prompts created")


In [ ]:
# 对照示例：看测试集第一条生成后的 prompt 与 completion
print("📝 Example item AFTER prompt creation:")
# 分隔线：标出 PROMPT 段
print(f"\n{'='*60}")
print("PROMPT:")
print(f"{'='*60}")
# 打印模型输入侧文本（含商品信息与前缀约定）
print(test[0].prompt)
# 分隔线：标出 COMPLETION 段（监督标签，通常是价格）
print(f"\n{'='*60}")
print("COMPLETION:")
print(f"{'='*60}")
print(test[0].completion)
print(f"{'='*60}")


In [ ]:
# 提示：统计「完整 prompt（含 completion）」的 token 数，对照 MAX_SEQ_LENGTH
print("Counting tokens in full prompts...")
# 对每条 Item 调用 count_prompt_tokens：量的是训练时真正进模型的序列长度
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]

# 完整序列的平均 token 数
avg_prompt_tokens = sum(prompt_token_counts) / len(prompt_token_counts)
# 完整序列的最大 token 数
max_prompt_tokens = max(prompt_token_counts)

# 打印统计（文案保持原样）
print(f"\n📊 Full Prompt Token Statistics:")
print(f"   Average: {avg_prompt_tokens:.1f} tokens")
print(f"   Maximum: {max_prompt_tokens} tokens")


In [ ]:
# 画「prompt + completion」长度分布，核对是否顶到 MAX_SEQ_LENGTH
plt.figure(figsize=(15, 6))
plt.title(f"Full Prompt Token Distribution: Avg {avg_prompt_tokens:.1f}, Max {max_prompt_tokens}")
# X：完整序列 token 数；Y：样本计数
plt.xlabel('Number of tokens (prompt + completion)')
plt.ylabel('Count')
# 直方图：金色柱，bins 与 summary 图一致便于对比
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
# 红虚线：训练时的最大序列长度上限
plt.axvline(config.MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'Max Seq Length: {config.MAX_SEQ_LENGTH}')
plt.legend()
plt.show()


## 步骤 4：上传至 HuggingFace Hub

把带 prompt/completion 的 train/val/test 推到 Hub，供后续微调笔记本直接 `load`。


In [ ]:
# 打印目标数据集仓库名（config.PROMPTS_DATASET_NAME）
print(f"Uploading to: {config.PROMPTS_DATASET_NAME}")
# 提醒：上传可能较慢，取决于数据量与网络
print("This may take a few minutes...")

# Item.push_prompts_to_hub：把三份列表序列化并 push 到指定 Hub 数据集
Item.push_prompts_to_hub(config.PROMPTS_DATASET_NAME, train, val, test)

# 成功提示 + 可在浏览器打开的数据集链接（URL 模板保持原样）
print(f"\n✅ Dataset uploaded successfully!")
print(f"\n🔗 View at: https://huggingface.co/datasets/{config.PROMPTS_DATASET_NAME}")


## 小结

✅ **数据准备完成！**

**我们做了什么：**

1. 从 HuggingFace Hub 加载商品数据集
2. 分析 summary 的 token 分布（平均大约 60 tokens）
3. 用约 110 tokens 的截止值（大约截断 5% 样本）
4. 生成结构化的 prompt–completion 对
5. 把训练数据上传到 HuggingFace Hub

**下一步：** 打开 `02_基座模型测试.ipynb` / `02_基线模型.ipynb` —— 在微调前先测基座 Llama，建立误差基线。
